# Gliquid Binary Fitting Demo

This notebook walks through the core workflow of the `gliquid` package:

1. Load elemental thermodynamic constants, digitized MPDS phase diagrams, and DFT convex hull entries from the Materials Project.
2. Run the Nelder-Mead optimizer to find liquid non-ideal mixing parameters for a binary system.
3. Batch fit and compare results across multiple systems.

In [ ]:
## UNCOMMENT THIS BLOCK FOR GOOGLE COLAB
# !git clone https://github.com/willwerj/gliquid_python.git
# %cd /content/gliquid_python
# !pip install -q .
# from pathlib import Path
# import gliquid.config as cfg
# cfg.set_data_dir(Path("/content/gliquid_python/data").resolve())

In [ ]:
import os
import gliquid.config as cfg

# By default cfg.project_root and cfg.data_dir are resolved from the installed package location.
# Override them here if your data lives elsewhere:
#   cfg.set_project_root("path/to/G_liquid")
#   cfg.set_data_dir("path/to/matrix_data")
#   cfg.set_dir_structure("nested")
print(f"Project root : {cfg.project_root}")
print(f"Data directory: {cfg.data_dir}")

# Materials Project API key — required for fetching DFT convex hull data.
# Replace with your own key from https://materialsproject.org/api
os.environ.setdefault("NEW_MP_API_KEY", "YOUR_API_KEY_HERE")

## 1 - Loading binary data

`gliquid.mpds` provides backend utilities for reading:
- **Measured elemental data** (melting enthalpy, melting temperature, boiling temperature) from JSON files in `cfg.data_dir`.
- **Digitized experimental phase diagrams** stored as MPDS-format JSON files (one per binary system).
- **DFT-calculated phase data** fetched from the Materials Project via `mp-api`.

MPDS phase diagrams are stored as SVG paths — essentially collections of curves that trace phase boundaries. The annotated figure below shows the anatomy of one such file.

<img src="../assets/digitized_pd_anatomy.png" style="max-width: 700px; width: 100%; height: auto;">


In [ ]:
import gliquid.api as api
import gliquid.mpds as mpds

# Elemental reference data loaded at import time from JSON files in cfg.data_dir
from gliquid.phase import UNARY
print("Melting temperatures (K)  :",
      {s: r.t_fusion for s, r in UNARY.elements.items() if r.t_fusion})
print("Melting enthalpies (J/mol):",
      {s: r.h_liq for s, r in UNARY.elements.items() if r.h_liq})
print()

# Load Cu-Mg MPDS data: returns the raw JSON, per-component thermodynamic constants,
# and the digitized liquidus curve as a list of [X (at. frac. Mg), T (K)] points.
cu_mg_json, (cu_mg_digitized_liquidus, is_partial) = mpds.load_mpds_data('Cu-Mg')
cu_mg_component_data = UNARY.component_data(['Cu', 'Mg'])

print("Component data (Elt: [H_fusion (J/mol), T_fusion (K)]):")
print(" ", cu_mg_component_data)
print(f"Digitized liquidus ({len(cu_mg_digitized_liquidus)} points):")
print(" ", [[round(x, 2), round(t, 2)] for x, t in cu_mg_digitized_liquidus], "...")
print()

# Identify solid phases from the MPDS JSON.
# 'lc' = line compound (no solid solubility), 'ss' = solid solution.
identified_cu_mg_phases = mpds.identify_mpds_phases(cu_mg_json)
print("Cu-Mg solid phases:")
for phase in identified_cu_mg_phases:
    print(" ", phase)


`get_dft_convexhull` queries the Materials Project and returns a pymatgen `PhaseDiagram` object representing the T = 0K DFT ground-state convex hull. Results are cached locally after the first call.


In [ ]:
from pymatgen.analysis.phase_diagram import PDPlotter

# Fetch (and cache) the DFT T=0K convex hull for Cu-Mg
cu_mg_dft_ch, _ = api.get_dft_convexhull('Cu-Mg', verbose=True)

cu_mg_pdp = PDPlotter(cu_mg_dft_ch)
fig = cu_mg_pdp.get_plot()
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", width=750, height=600)
fig.show()


## 2 - Fitting the liquid free energy for a single binary system

`fit_parameters()` finds the liquid non-ideal mixing parameters by the DFT-referenced liquidus to the digitized one. 
It:
1. Identifies invariant points (eutectics, peritectics, congruent melting) from the liquidus and solid-phase data.
2. Filters to those that have matching DFT compounds.
3. Ranks constraint combinations by viability and runs Nelder-Mead optimization for each.

The diagrams below illustrate how invariant points on the phase diagram translate into thermodynamic constraints on the mixing parameters. We first build the Cu-Mg object by hand, supplying the invariant points ourselves, then show the `from_cache` shortcut that performs the same setup automatically.

<img src="../assets/invariant_points_and_reactions.png" style="max-width: 100%; height: 480px; object-fit: contain; object-position: left center; display: block; margin: 0 !important;">


### Manual `BinaryLiquid` initialization 

We require several sources of data to regress the non-ideal mixing parameters for a system. Without cached data, these need to be input manually, as shown below.

In [ ]:
# 1) Define elements and elemental reference states (enthalpies in J/mol, temperatures in K).
#    The package models element references as ComponentRef/Phase objects (gliquid.phase):
#    each phase carries its cumulative enthalpy/entropy relative to the DFT ground state.
from gliquid.phase import ComponentRef, Phase

def liquid_ref(symbol, h_fusion, t_fusion, t_vaporization):
    """ComponentRef with just a liquid (fusion) and gas (boiling) reference — S_fus = H_fus / T_fus."""
    return ComponentRef(symbol, [
        Phase(phase_type='liquid', t_transition=t_fusion,
                 enthalpy=h_fusion, entropy=h_fusion / t_fusion, delta_h=h_fusion),
        Phase(phase_type='gas', t_transition=t_vaporization),
    ])

cu_mg_components = ['Cu', 'Mg']
cu_mg_component_data = {
    'Cu': liquid_ref('Cu', h_fusion=13260.0, t_fusion=1357.77, t_vaporization=2835.15),
    'Mg': liquid_ref('Mg', h_fusion=8480.0,  t_fusion=923.0,  t_vaporization=1363.15),
}

# 2) Define the measured liquidus curve as a list of [X (at. frac. Mg), T (K)] points.
cu_mg_liquidus = [[0.0, 1356.77], [0.01, 1339.78], [0.03, 1322.09], [0.04, 1303.81], [0.06, 1284.96], [0.07, 1265.54],
                  [0.08, 1245.58], [0.1, 1225.08], [0.11, 1204.06], [0.12, 1182.53], [0.13, 1160.51], [0.15, 1138.01],
                  [0.16, 1115.04], [0.17, 1091.62], [0.18, 1067.76], [0.2, 1043.47], [0.21, 1018.77], [0.22, 994.15],
                  [0.23, 1002.94], [0.24, 1011.81], [0.25, 1020.36], [0.26, 1028.56], [0.26, 1036.36], [0.27, 1043.74],
                  [0.28, 1050.66], [0.28, 1057.08], [0.29, 1062.98], [0.3, 1068.31], [0.3, 1073.05], [0.31, 1077.15], 
                  [0.31, 1080.6], [0.32, 1083.34], [0.32, 1085.35], [0.33, 1086.58], [0.33, 1087.01], [0.35, 1085.6],
                  [0.36, 1081.49], [0.38, 1074.87], [0.39, 1065.91], [0.41, 1054.83], [0.42, 1041.8], [0.43, 1027.0], 
                  [0.45, 1010.62], [0.46, 992.85], [0.48, 973.88], [0.49, 953.87], [0.5, 933.03], [0.52, 911.53],
                  [0.53, 889.57], [0.55, 867.31], [0.56, 844.96], [0.57, 822.94], [0.58, 824.57], [0.59, 826.2], 
                  [0.59, 827.74], [0.6, 829.19], [0.6, 830.56], [0.61, 831.83], [0.61, 833.0], [0.62, 834.07],
                  [0.62, 835.04], [0.63, 835.91], [0.64, 836.67], [0.64, 837.32], [0.65, 837.86], [0.65, 838.28],
                  [0.66, 838.59], [0.66, 838.77], [0.67, 838.84], [0.68, 838.46], [0.69, 837.35], [0.7, 835.54],
                  [0.72, 833.07], [0.73, 829.97], [0.74, 826.28], [0.75, 822.02], [0.76, 817.25], [0.77, 811.99],
                  [0.79, 806.27], [0.8, 800.13], [0.81, 793.61], [0.82, 786.73], [0.83, 779.55], [0.84, 772.08], 
                  [0.85, 764.37], [0.86, 756.71], [0.87, 767.98], [0.87, 779.43], [0.88, 790.69], [0.89, 801.77],
                  [0.9, 812.63], [0.9, 823.28], [0.91, 833.71], [0.92, 843.89], [0.93, 853.84], [0.94, 863.53],
                  [0.95, 872.95], [0.96, 882.1], [0.96, 890.96], [0.97, 899.53], [0.98, 907.79], [0.99, 915.74], 
                  [1.0, 923.26]] 

# 3) Define the solid phases in the Cu-Mg system, and any liquid-solid invariant reactions which are known
from gliquid.hull_editor import ConvexHullEditor
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
el_energy = {str(el): ref.energy_per_atom for el, ref in cu_mg_dft_ch.el_refs.items()}
editor = ConvexHullEditor(PhaseDiagram([PDEntry(c, el_energy[c], name=c) for c in cu_mg_components]))
editor.add_entry('MgCu2', -0.143, name='MgCu2')   # formation energy in eV/atom (from DFT)
editor.add_entry('Mg2Cu', -0.106, name='Mg2Cu')
editor.display()   # interactive: this can be used to add/modify entries instead of programmatically 
manual_dft_ch = editor.apply()

cu_mg_invariants = [
    {'type': 'cmp', 'comp': 1/3, 'temp': 1092.0, 'phases': ['MgCu2'], 'phase_comps': [1/3]},
    {'type': 'cmp', 'comp': 2/3, 'temp': 841.0,  'phases': ['Mg2Cu'], 'phase_comps': [2/3]},
]

In [ ]:
import numpy as np
from gliquid.binary import BinaryLiquid, BLPlotter, build_phases_from_chull
from gliquid.solution import RKPolyExp, SolutionModel, x_vals as _x_vals, t_sym

param_format = 'comb-exp'  # options: 'linear', 'combined', 'comb-exp'
phases = build_phases_from_chull(manual_dft_ch, cu_mg_components, cu_mg_component_data)
eqs = SolutionModel(
    ('Cu', 'Mg'),
    (cu_mg_component_data['Cu'].gibbs_ref_expr(t_sym),
     cu_mg_component_data['Mg'].gibbs_ref_expr(t_sym)),
    {(0, 1): RKPolyExp(param_format)}).binary_eqs()
hull_points = np.array([[p.fraction_in(cu_mg_components)[0], p.enthalpy]
                        for p in phases if p.composition is not None])
eqs['h_hull_interp'] = np.interp(_x_vals[1:-1], hull_points[:, 0], hull_points[:, 1])

manual_system = BinaryLiquid(
    'Cu-Mg', cu_mg_components,
    component_data=cu_mg_component_data,
    digitized_liq=cu_mg_digitized_liquidus,  
    dft_ch=manual_dft_ch,
    phases=phases,
    param_format=param_format,
    temp_range=[cu_mg_json['temp'][0] + 273.15, cu_mg_json['temp'][1] + 273.15],
    eqs=eqs,
    invariants=cu_mg_invariants
)

manual_fit = manual_system.fit_parameters(verbose=True, n_opts=3)
manual_best = min(manual_fit, key=lambda x: x.get('mae', float('inf')), default={})
print("\n--- Cu-Mg manual fit ---")
print("  params:", [round(p, 3) for p in manual_system.get_params()])
print(f"  MAE: {manual_best.get('mae', float('nan')):.2f} K")

BLPlotter(manual_system).show('fit+liq')   # fitted liquidus vs. digitized experimental data

### `from_cache` intialization

`BinaryLiquid.from_cache()` performs the same assembly automatically: it loads the elemental constants, fetches the DFT convex hull, reads the MPDS phase diagram, and detects the invariant points for you. This approach relies on having cached MPDS phase diagram data, or an MPDS API key

We can also generate plots to display other information on the fitted parameters and resulting DFT-referenced phase diagram


In [ ]:
from gliquid.binary import BinaryLiquid, BLPlotter
import matplotlib.pyplot as plt

param_format = 'comb-exp'  # options: 'linear', 'combined', 'comb-exp'

cu_mg_system = BinaryLiquid.from_cache('Cu-Mg', param_format=param_format)

# n_opts controls how many constraint combinations are explored; higher values are slower but more thorough
fit_results = cu_mg_system.fit_parameters(verbose=True, n_opts=5)
best_fit = min(fit_results, key=lambda x: x.get('mae', float('inf')), default={})

print("\n--- Cu-Mg fitting results ---")
for field, value in best_fit.items():
    print(f"  {field.upper():12s}: {value:.3f}" if isinstance(value, float) else f"  {field.upper():12s}: {value}")

cu_mg_plotter = BLPlotter(cu_mg_system)
cu_mg_plotter.show('fit+liq')   # Fitted liquidus vs. digitized experimental data
cu_mg_plotter.get_plot('pc')        # Compare the low temperature phase diagram with the DFT convex hull
plt.show()
cu_mg_plotter.show('ch+g')      # T=0K DFT convex hull with overlaid liquid G curves
cu_mg_plotter.show('nmp')       # Nelder-Mead optimization path through parameter space


## 3 - Batch fitting across multiple binary systems

Results are cached to a JSON file after the first run. Delete the cache file to force a re-fit.


In [ ]:
import json
import numpy as np
from pathlib import Path

systems_to_fit = ['C-Nb', 'Cr-Eu', 'Ag-V', 'Al-Cu']
cache_file = str(Path.cwd().parent / 'data' / f'fit_results_cache_{param_format}.json')

if os.path.exists(cache_file):
    with open(cache_file) as f:
        fitting_results = json.load(f)
    # Restore numpy arrays (not preserved by JSON serialization)
    for sys_results in fitting_results.values():
        sys_results['nmpath'] = np.array(sys_results.get('nmpath', []))
    print(f"Loaded cached results for: {list(fitting_results)}")
else:
    fitting_results = {}
    for sys_name in systems_to_fit:
        bl = BinaryLiquid.from_cache(sys_name, param_format=param_format)
        attempts = bl.fit_parameters(verbose=False, n_opts=3)
        fitting_results[sys_name] = min(attempts, key=lambda x: x.get('mae', float('inf')), default={})

    # Convert ndarray → list before serializing
    serializable = {k: {**v, 'nmpath': v.get('nmpath', np.array([])).tolist()}
                    for k, v in fitting_results.items()}
    with open(cache_file, 'w') as f:
        json.dump(serializable, f)
    print(f"Saved results to {cache_file}")


In [ ]:
import matplotlib.pyplot as plt

for sys_name, best_results in fitting_results.items():
    if not all(k in best_results for k in ['L0_a', 'L0_b', 'L1_a', 'L1_b']):
        print(f"{sys_name}: incomplete results, skipping")
        continue

    bl = BinaryLiquid.from_cache(sys_name, param_format=param_format)
    bl.update_params((best_results['L0_a'], best_results['L0_b'], best_results['L1_a'], best_results['L1_b']))
    bl.nmpath = np.array(best_results.get('nmpath', []))

    params_str = [float(f'{p:.2f}') for p in bl.get_params()]
    print(f"\n{sys_name}  params={params_str}  MAE={best_results['mae']:.1f}K  MAPE={best_results['mape']:.2f}%")

    blp = BLPlotter(bl)
    # plt.show() is used here instead of blp.show() to ensure inline rendering in Jupyter
    blp.get_plot('nmp', plot_a_params=True); plt.show()
    blp.show('fit+liq')
    blp.show('ch+g')
